## Imports

In [101]:
import yfinance as yf
import pandas as pd
import os
import boto3
from datetime import *
from dotenv import load_dotenv
import numpy as np
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import pyarrow

## Conection AWS S3

In [102]:
load_dotenv()

ACCESS_KEY = os.getenv('ACCESS_KEY')
SECRET_KEY = os.getenv('SECRET_KEY')

client = boto3.client(
    's3',
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY
)

# Bronze Layer

In [103]:
df = yf.download(['AAPL', 'NVDA'], period='max', interval='1d', multi_level_index=False, actions=True)

[*********************100%***********************]  2 of 2 completed


In [104]:
today = datetime.now().strftime('%d-%m-%y')
os.makedirs('input', exist_ok=True)
df.to_csv(f'input/stocks_{today}.csv', index=False)

In [105]:
# Broze layer of data saved in S3 bucket.
client.upload_file(Filename=f'input/stocks_{today}.csv', Bucket='ace-low', Key=f'finance/stocks_{today}.csv')

# Silver Layer

In [106]:
df = df.stack(level='Ticker').reset_index()
df.columns.name = None
df.head(8)

C:\Users\Pichau\AppData\Local\Temp\ipykernel_11404\216972287.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.stack(level='Ticker').reset_index()


,Date,Ticker,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,1980-12-12,AAPL,0.098298,0.0,0.098725,0.098298,0.098298,0.0,469033600.0
1,1980-12-15,AAPL,0.093169,0.0,0.093597,0.093169,0.093597,0.0,175884800.0
2,1980-12-16,AAPL,0.086331,0.0,0.086758,0.086331,0.086758,0.0,105728000.0
3,1980-12-17,AAPL,0.088468,0.0,0.088895,0.088468,0.088468,0.0,86441600.0
4,1980-12-18,AAPL,0.091033,0.0,0.091460,0.091033,0.091033,0.0,73449600.0
5,1980-12-19,AAPL,0.096588,0.0,0.097015,0.096588,0.096588,0.0,48630400.0
6,1980-12-22,AAPL,0.101289,0.0,0.101717,0.101289,0.101289,0.0,37363200.0
7,1980-12-23,AAPL,0.105563,0.0,0.105991,0.105563,0.105563,0.0,46950400.0


## EDA and Data cleaning

In [107]:
df.columns = (df.columns.str.upper().str.replace(' ', '_'))
df.head(5)

,DATE,TICKER,CLOSE,DIVIDENDS,HIGH,LOW,OPEN,STOCK_SPLITS,VOLUME
0,1980-12-12,AAPL,0.098298,0.0,0.098725,0.098298,0.098298,0.0,469033600.0
1,1980-12-15,AAPL,0.093169,0.0,0.093597,0.093169,0.093597,0.0,175884800.0
2,1980-12-16,AAPL,0.086331,0.0,0.086758,0.086331,0.086758,0.0,105728000.0
3,1980-12-17,AAPL,0.088468,0.0,0.088895,0.088468,0.088468,0.0,86441600.0
4,1980-12-18,AAPL,0.091033,0.0,0.091460,0.091033,0.091033,0.0,73449600.0


In [108]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18289 entries, 0 to 18288
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   DATE          18289 non-null  datetime64[ns]
 1   TICKER        18289 non-null  object        
 2   CLOSE         18289 non-null  float64       
 3   DIVIDENDS     18289 non-null  float64       
 4   HIGH          18289 non-null  float64       
 5   LOW           18289 non-null  float64       
 6   OPEN          18289 non-null  float64       
 7   STOCK_SPLITS  18289 non-null  float64       
 8   VOLUME        18289 non-null  float64       
dtypes: datetime64[ns](1), float64(7), object(1)
memory usage: 1.3+ MB


In [109]:
# sorting values by highest date
df = df.sort_values(by='DATE', ascending=False)

df['VARIATION_PAST_DAY'] = df['CLOSE'].diff()

df['PERCENT_VARIATION_PAST_DAY'] = df['CLOSE'].pct_change()

df['GAP_PAST_DAY'] = (df['OPEN'] - df['CLOSE']).shift(1)

df['FLAG_DAY'] = np.where(
    df['OPEN'] < df['CLOSE'],
    'POSITIVE',
    np.where(
        df['OPEN'] == df['CLOSE'],
        'EQUAL',
        'NEGATIVE'
    )
)

df['KEY'] = df['TICKER'].astype(str) + "-" + df['DATE'].dt.strftime('%Y-%m-%d')

df['INSERT_DATE'] = datetime.now()
df['INSERT_DATE'] = df['INSERT_DATE'].dt.strftime('%Y-%m-%d %H:%M:%S')
df['DATE'] = df['DATE'].dt.strftime('%Y-%m-%d %H:%M:%S')

## Connect Snowflake

In [110]:
## Get env keys
load_dotenv()
USER_SNOW = os.getenv('USER_SNOW')
PASSWORD_SNOW = os.getenv('PASSWORD_SNOW')
ACCOUNT_SNOW = os.getenv('ACCOUNT_SNOW')

con_snow = snowflake.connector.connect(
    user=USER_SNOW,
    password=PASSWORD_SNOW,
    account=ACCOUNT_SNOW
)

In [111]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18289 entries, 18288 to 0
Data columns (total 15 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   DATE                        18289 non-null  object 
 1   TICKER                      18289 non-null  object 
 2   CLOSE                       18289 non-null  float64
 3   DIVIDENDS                   18289 non-null  float64
 4   HIGH                        18289 non-null  float64
 5   LOW                         18289 non-null  float64
 6   OPEN                        18289 non-null  float64
 7   STOCK_SPLITS                18289 non-null  float64
 8   VOLUME                      18289 non-null  float64
 9   VARIATION_PAST_DAY          18288 non-null  float64
 10  PERCENT_VARIATION_PAST_DAY  18288 non-null  float64
 11  GAP_PAST_DAY                18288 non-null  float64
 12  FLAG_DAY                    18289 non-null  object 
 13  KEY                         18289 no

### Create DW and table

In [112]:
con_snow.cursor().execute("CREATE WAREHOUSE IF NOT EXISTS ace")
con_snow.cursor().execute("CREATE DATABASE IF NOT EXISTS ace_low")
con_snow.cursor().execute("USE DATABASE ace_low")
con_snow.cursor().execute("CREATE SCHEMA IF NOT EXISTS stocks")

con_snow.cursor().execute("""
                          CREATE TABLE IF NOT EXISTS stocks.stock_market(
                            key string PRIMARY KEY,
                            date datetime NOT NULL,
                            ticker string,
                            open float,
                            low float,
                            high float,
                            close float,
                            stock_splits float,
                            dividends float,
                            volume float,
                            variation_past_day float,
                            percent_variation_past_day float,
                            gap_past_day float,
                            flag_day string,
                            insert_date datetime
                          )
                        """)

In [113]:
existing_keys = con_snow.cursor().execute("SELECT KEY FROM ace_low.stocks.stock_market").fetch_pandas_all()

In [114]:
df_new = df[~df['KEY'].isin(existing_keys['KEY'])]


if not df_new.empty:
    success, nchunks, nrows, _ = write_pandas(
        conn=con_snow,
        df=df_new,
        schema='STOCKS',
        table_name='STOCK_MARKET'
    )
    print(f"{nrows} rows inserted!")
else:
    print("Nothing new to insert in the Table!")

Nothing new to insert in the Table!
